<a href="https://colab.research.google.com/github/roywe/book_generator_tom/blob/master/notebooks/text2stl_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TOM — Tactile Hebrew Storybook Generator

Step-by-step pipeline for generating one tactile book page:
1. Enter a Hebrew word → Stable Diffusion generates a line-art image
2. Add nikud (vowel marks) → convert to Braille
3. Export three DXF files (image, Hebrew text, Braille)
4. Merge into a single 3D-printable STL

## Setup
Clone the repo and install dependencies. Run this cell once per Colab session.

In [ ]:
import os

if not os.path.exists('book_generator_tom'):
    !git clone https://github.com/roywe/book_generator_tom.git

%cd book_generator_tom
!pip install -r requirements.txt -q

## Imports

In [ ]:
%matplotlib inline
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from diffusers import AutoPipelineForText2Image
from pathlib import Path

from src.config import cfg
from src.language_funcs import hebrew_translator, add_nikud, convert_to_braille
from src.image_funcs import (
    ensure_font,
    process_image_to_dxf,
    generate_hebrew_text_dxf,
    generate_braille_dxf_from_text,
    plot_dxf,
)
from src.dxf_3d import create_one_page_stl_from_dxf

ensure_font()
os.makedirs('outputs', exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Step 1 — Load Stable Diffusion model

In [ ]:
model_id = cfg['stable_diffusion']['model_id']
dtype = torch.float16 if device == 'cuda' else torch.float32

print(f'Loading {model_id}...')
pipe = AutoPipelineForText2Image.from_pretrained(model_id, torch_dtype=dtype).to(device)
print('Model ready.')

## Step 2 — Generate image from Hebrew input

In [ ]:
hebrew_prompt = input('הכנס/י את המילה שתרצי/שתרצה בספר: ')
picture_type  = input('הכנס/י את הקטגוריה של התמונה (פרי, ירק, מספר, אות): ')

eng_desc  = hebrew_translator(hebrew_prompt)
eng_class = hebrew_translator(picture_type)

style_prompt = (
    "Simple child's drawing, 2D flat design, outlines only, single thin black pen, "
    "minimalistic, continuous single pen draw, broad strokes, white background."
)
negative_prompt = (
    'background, scenery, environment, extra items, shading, shadows, gradients, '
    'grayscale, fine lines, intricate details, realistic texture, dots, 3D, depth, '
    'perspective, messy lines, broken lines.'
)
final_prompt = (
    f'A single isolated {eng_desc} centered on a white background, '
    f'classified as {eng_class}. {style_prompt}'
)

sd_cfg = cfg['stable_diffusion']
image = pipe(
    prompt=final_prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=sd_cfg['inference_steps'],
    guidance_scale=sd_cfg['guidance_scale'],
).images[0]

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis('off')
plt.title(f'Generated: {hebrew_prompt}')
plt.show()

## Step 3 — Add nikud and convert to Braille

`add_nikud()` prompts interactively for vowel marks (dagesh, holam, shuruk, etc.).

In [ ]:
hebrew_with_nikud = add_nikud(hebrew_prompt)
braille = convert_to_braille(hebrew_with_nikud)

print(f'Hebrew with nikud : {hebrew_with_nikud}')
print(f'Braille            : {braille}')

## Step 4 — Export three DXF files

In [ ]:
word_slug = hebrew_prompt.replace(' ', '_')

dxf_image   = f'outputs/{word_slug}_image.dxf'
dxf_text    = f'outputs/{word_slug}_text.dxf'
dxf_braille = f'outputs/{word_slug}_braille.dxf'

process_image_to_dxf(np.array(image), dxf_image)
generate_hebrew_text_dxf(hebrew_with_nikud, dxf_text)
generate_braille_dxf_from_text(braille, dxf_braille)

print('DXF files saved:')
for p in [dxf_image, dxf_text, dxf_braille]:
    size = os.path.getsize(p) if os.path.exists(p) else 0
    print(f'  {p}  ({size:,} bytes)')

## Step 5 — Build STL

Merges the three DXF layers into a single 150×150mm 3D-printable plate.

In [ ]:
stl_path = f'outputs/{word_slug}.stl'

create_one_page_stl_from_dxf(
    txt_dxf=Path(dxf_text),
    braille_dxf=Path(dxf_braille),
    image_dxf=Path(dxf_image),
    output=Path(stl_path),
)

print(f'STL saved: {stl_path}  ({os.path.getsize(stl_path):,} bytes)')

## Preview DXFs

In [ ]:
for path, title in [(dxf_image, 'Image'), (dxf_text, 'Hebrew text'), (dxf_braille, 'Braille')]:
    print(f'--- {title} ---')
    plot_dxf(path)